In [2]:
!pip install dash

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\kamel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import os
import pandas as pd
import numpy as np

# 1. مسار مجلد الملفات
DATA_FOLDER = r"D:\\HVIA\\project_one\\Sharnouby_Enterprise (3)\\data"  # غيّر المسار إلى اسم المجلد لديك

dfs = {}
EXPECTED_SCHEMAS = {}

# 2. قراءة جميع ملفات CSV تلقائياً
for file_name in os.listdir(DATA_FOLDER):
    if file_name.endswith('.csv'):
        table_name = os.path.splitext(file_name)[0]
        file_path = os.path.join(DATA_FOLDER, file_name)
        
        # قراءة الملف
        df = pd.read_csv(file_path)
        
        # تخزين البيانات والـ Schema
        dfs[table_name] = df
        EXPECTED_SCHEMAS[table_name] = df.columns.tolist()

print(f" length  {len(dfs)}  CSV .\n")

 length  20  CSV .



In [5]:
def preprocess_pipeline(data_dict):
    cleaned_dfs = {}
    
    for name, df in data_dict.items():
        # نسخة للحفاظ على البيانات الأصلية
        cdf = df.copy()
        
        # 1. تنظيف أسماء الأعمدة
        cdf.columns = cdf.columns.str.strip()
        
        # 2. إزالة الأسطر المكررة بالكامل
        initial_rows = len(cdf)
        cdf = cdf.drop_duplicates()
        dropped_dups = initial_rows - len(cdf)
        
        # 3. تنظيف القيم النصية (strip whitespace)
        str_cols = cdf.select_dtypes(include=['object']).columns
        for col in str_cols:
            cdf[col] = cdf[col].astype(str).str.strip()
            
        # 4. تحويل التواريخ تلقائياً
        date_cols = [c for c in cdf.columns if 'Date' in c or 'Month' in c or 'Snapshot' in c]
        for c in date_cols:
            cdf[c] = pd.to_datetime(cdf[c], errors='coerce')
            
        cleaned_dfs[name] = cdf
        
        # طباعة ملخص سريع لكل جدول
        null_count = cdf.isnull().sum().sum()
        print(f"📊 [{name}] -> rows: {len(cdf)} | columns: {len(cdf.columns)} | missing values: {null_count} | duplicates removed: {dropped_dups}")
        
    return cleaned_dfs

# تشغيل الـ Preprocessing
cleaned_dfs = preprocess_pipeline(dfs)

📊 [Business_Units] -> rows: 5 | columns: 5 | missing values: 0 | duplicates removed: 0
📊 [Case_Briefs] -> rows: 10 | columns: 4 | missing values: 0 | duplicates removed: 0
📊 [Customers] -> rows: 30000 | columns: 9 | missing values: 0 | duplicates removed: 0
📊 [Customer_Service] -> rows: 6500 | columns: 9 | missing values: 0 | duplicates removed: 0
📊 [Data_Dictionary] -> rows: 19 | columns: 4 | missing values: 0 | duplicates removed: 0
📊 [Group_Operations] -> rows: 180 | columns: 7 | missing values: 0 | duplicates removed: 0
📊 [Inventory_History] -> rows: 6372 | columns: 19 | missing values: 0 | duplicates removed: 0
📊 [Locations] -> rows: 5 | columns: 9 | missing values: 0 | duplicates removed: 0
📊 [Machine_Downtime] -> rows: 2800 | columns: 8 | missing values: 0 | duplicates removed: 0
📊 [Marketing] -> rows: 420 | columns: 15 | missing values: 0 | duplicates removed: 0
📊 [Materials] -> rows: 12 | columns: 6 | missing values: 0 | duplicates removed: 0
📊 [Printing_Batches] -> rows: 1900

In [8]:
# 1. تشخيص القيم المفقودة في Sales_Transactions لمعرفة سبب الـ 171,457 قيمة مفقودة
sales_df = cleaned_dfs["Sales_Transactions"]
print(" Details of Missing Values in Sales_Transactions:")
missing_sales = sales_df.isnull().sum()
print(missing_sales[missing_sales > 0])
print("=" * 60)

# 2. استخراج المفاتيح والعلاقات تلقائياً بدون كتابة أي أسماء يدوياً
def auto_discover_relationships(data_dict):
    primary_keys = {}
    
    # أ) اكتشاف المفاتيح الأساسية المحتملة في كل جدول
    for table_name, df in data_dict.items():
        pks = []
        for col in df.columns:
            # المفتاح الأساسي عادة معرف/رمز ويحتوي على قيم فريدة وغير مفقودة
            if ('ID' in col or 'Key' in col or 'No' in col or 'Code' in col or 'Date' in col) and df[col].notnull().all():
                if df[col].is_unique:
                    pks.append(col)
        primary_keys[table_name] = pks

    discovered_fks = []
    
    # ب) اكتشاف المفاتيح الخارجية والتحقق من سلامة الربط تلقائياً
    for child_table, child_df in data_dict.items():
        for col in child_df.columns:
            # نبحث عن العمود في الجداول الأخرى لمعرفة ما إذا كان مفتاحاً رئيسياً هناك
            for parent_table, pks in primary_keys.items():
                if child_table != parent_table and col in pks:
                    # فحص القيم غير المكتملة أو اليتيمة (Orphans)
                    child_vals = child_df[col].dropna()
                    parent_vals = data_dict[parent_table][col].dropna()
                    
                    if len(child_vals) > 0:
                        is_subset = child_vals.isin(parent_vals)
                        match_pct = (is_subset.sum() / len(child_vals)) * 100
                        orphans_count = len(child_vals[~is_subset])
                        
                        discovered_fks.append({
                            "Child Table": child_table,
                            "Foreign Key": col,
                            "Parent Table": parent_table,
                            "Match Pct (%)": round(match_pct, 2),
                            "Orphan Rows": orphans_count
                        })
                        
    return primary_keys, pd.DataFrame(discovered_fks)

# تشغيل الاكتشاف التلقائي
detected_pks, fk_df = auto_discover_relationships(cleaned_dfs)

print(" (Primary Keys):")
for t, pks in detected_pks.items():
    if pks:
        print(f"  - {t}: {pks}")

print("\n (Foreign Keys):")
print(fk_df.to_string(index=False))

 Details of Missing Values in Sales_Transactions:
Return_Date    171457
dtype: int64
 (Primary Keys):
  - Case_Briefs: ['Card_No']
  - Customers: ['Customer_ID']
  - Customer_Service: ['Ticket_ID']
  - Locations: ['Location_ID']
  - Machine_Downtime: ['Downtime_ID']
  - Marketing: ['Campaign_ID']
  - Materials: ['Material_ID']
  - Printing_Batches: ['Print_Batch_ID']
  - Production_Orders: ['Production_Order_ID']
  - Product_Master: ['Product_ID']
  - Purchase_Orders: ['PO_ID']
  - Quality: ['QC_ID']
  - Sales_Transactions: ['Row_ID', 'Invoice_ID']
  - Suppliers: ['Supplier_ID']
  - Web_Traffic_Daily: ['Date']

 (Foreign Keys):
       Child Table Foreign Key      Parent Table  Match Pct (%)  Orphan Rows
 Inventory_History Location_ID         Locations          100.0            0
 Inventory_History  Product_ID    Product_Master          100.0            0
  Machine_Downtime        Date Web_Traffic_Daily          100.0            0
         Marketing  Product_ID    Product_Master        

In [9]:
print("=" * 60)
print(" (EDA Summary)")
print("=" * 60)

# 1. تحليل الأداء المالي والمبيعات (Sales & Financials)
sales_df = cleaned_dfs["Sales_Transactions"]
total_gross_sales = sales_df["Gross_Sales_EGP"].sum()
total_net_sales = sales_df["Net_Sales_EGP"].sum()
total_profit = sales_df["Gross_Profit_EGP"].sum()
overall_margin = (total_profit / total_net_sales) * 100 if total_net_sales else 0
return_rate = (sales_df["Returned"].sum() / len(sales_df)) * 100

print(f"\n 1. Sales and financial performance:")
print(f"  - Total Gross Sales: {total_gross_sales:,.2f} EGP")
print(f"  - Total Net Sales:   {total_net_sales:,.2f} EGP")
print(f"  - Total Gross Profit:   {total_profit:,.2f} EGP")
print(f"  - Average Gross Margin %:    {overall_margin:.2f}%")
print(f"  - Return Rate %:        {return_rate:.2f}%")

# 2. تحليل حركة المخزون وسلسلة الإمداد (Inventory & Supply Chain)
if "Inventory_History" in cleaned_dfs:
    inv_df = cleaned_dfs["Inventory_History"]
    avg_days_cover = inv_df["Days_of_Cover"].mean()
    total_stockout_days = inv_df["Stockout_Days_In_Month"].sum()
    print(f"\n 2. Inventory and supply chain:")
    print(f"  - Average Days of Cover: {avg_days_cover:.1f} days")
    print(f"  - Total Stockout Days: {total_stockout_days:,} days")

# 3. تحليل كفاءة التصنيع والطباعة (Production & Quality)
if "Production_Orders" in cleaned_dfs and "Quality" in cleaned_dfs:
    prod_df = cleaned_dfs["Production_Orders"]
    qual_df = cleaned_dfs["Quality"]
    
    avg_prod_efficiency = prod_df["Efficiency_Pct"].mean()
    total_rejected_qty = prod_df["Rejected_Qty"].sum()
    total_quality_loss = qual_df["Estimated_Loss_EGP"].sum()
    
    print(f"\n 3. Production and Quality:")
    print(f"  - Average Production Efficiency:             {avg_prod_efficiency:.2f}%")
    print(f"  - Total Rejected Quantity:    {total_rejected_qty:,} pieces")
    print(f"  - Total Quality Loss (Estimated Loss): {total_quality_loss:,.2f} EGP")

# 4. تحليل التسويق والزيارات الإلكترونية (Marketing & Web Traffic)
if "Marketing" in cleaned_dfs and "Web_Traffic_Daily" in cleaned_dfs:
    mkt_df = cleaned_dfs["Marketing"]
    web_df = cleaned_dfs["Web_Traffic_Daily"]
    
    total_mkt_spend = mkt_df["Spend_EGP"].sum()
    total_attr_rev = mkt_df["Attributed_Revenue_EGP"].sum()
    roas = total_attr_rev / total_mkt_spend if total_mkt_spend else 0
    avg_conversion = web_df["Conversion_Rate"].mean()
    
    print(f"\n 4. Marketing and Web Traffic:")
    print(f"  - Total Marketing Spend:             {total_mkt_spend:,.2f} EGP")
    print(f"  - Attributed Revenue:       {total_attr_rev:,.2f} EGP")
    print(f"  - Return on Ad Spend (ROAS):    {roas:.2f}x")
    print(f"  - Average Conversion Rate: {avg_conversion:.2f}%")

# 5. تحليل خدمة العملاء (Customer Service)
if "Customer_Service" in cleaned_dfs:
    cs_df = cleaned_dfs["Customer_Service"]
    avg_csat = cs_df["CSAT_1_to_5"].mean()
    avg_res_hours = cs_df["Resolution_Hours"].mean()
    resolved_pct = (cs_df["Resolved"].astype(str).str.lower().isin(["true", "1", "yes"]).sum() / len(cs_df)) * 100
    
    print(f"\n🎧 5. Customer Service:")
    print(f"  - Average Customer Satisfaction (CSAT):            {avg_csat:.2f} / 5.0")
    print(f"  - Average Resolution Time (Resolution Hours):   {avg_res_hours:.1f} hours")
    print(f"  - Resolution Rate (Resolved %):   {resolved_pct:.2f}%")

print("\n" + "=" * 60)

 (EDA Summary)


TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [11]:
import pandas as pd
import numpy as np

print("=" * 60)
print("  (EDA Summary)")
print("=" * 60)

# 1. تحليل الأداء المالي والمبيعات (Sales & Financials)
if "Sales_Transactions" in cleaned_dfs:
    sales_df = cleaned_dfs["Sales_Transactions"].copy()
    
    # تحويل الأرقام بأمان
    num_cols = ["Gross_Sales_EGP", "Net_Sales_EGP", "Gross_Profit_EGP"]
    for col in num_cols:
        sales_df[col] = pd.to_numeric(sales_df[col], errors='coerce')
        
    total_gross_sales = sales_df["Gross_Sales_EGP"].sum()
    total_net_sales = sales_df["Net_Sales_EGP"].sum()
    total_profit = sales_df["Gross_Profit_EGP"].sum()
    overall_margin = (total_profit / total_net_sales) * 100 if total_net_sales else 0
    
    # تحويل Returned لبوليان بأمان
    returned_series = sales_df["Returned"].astype(str).str.upper().isin(["TRUE", "1", "YES"])
    return_rate = (returned_series.sum() / len(sales_df)) * 100

    print(f"\n 1. Sales and financial performance:")
    print(f"  - Total Gross Sales: {total_gross_sales:,.2f} EGP")
    print(f"  - Total Net Sales:   {total_net_sales:,.2f} EGP")
    print(f"  - Total Gross Profit:   {total_profit:,.2f} EGP")
    print(f"  - Average Gross Margin %:    {overall_margin:.2f}%")
    print(f"  - Return Rate %:        {return_rate:.2f}%")

# 2. تحليل حركة المخزون وسلسلة الإمداد (Inventory & Supply Chain)
if "Inventory_History" in cleaned_dfs:
    inv_df = cleaned_dfs["Inventory_History"].copy()
    inv_df["Days_of_Cover"] = pd.to_numeric(inv_df["Days_of_Cover"], errors='coerce')
    inv_df["Stockout_Days_In_Month"] = pd.to_numeric(inv_df["Stockout_Days_In_Month"], errors='coerce')
    
    avg_days_cover = inv_df["Days_of_Cover"].mean()
    total_stockout_days = inv_df["Stockout_Days_In_Month"].sum()
    print(f"\n 2. Inventory and supply chain:")
    print(f"  - Average Days of Cover: {avg_days_cover:.1f} days")
    print(f"  - Total Stockout Days: {total_stockout_days:,} days")

# 3. تحليل كفاءة التصنيع والجودة (Production & Quality)
if "Production_Orders" in cleaned_dfs and "Quality" in cleaned_dfs:
    prod_df = cleaned_dfs["Production_Orders"].copy()
    qual_df = cleaned_dfs["Quality"].copy()
    
    prod_df["Efficiency_Pct"] = pd.to_numeric(prod_df["Efficiency_Pct"], errors='coerce')
    prod_df["Rejected_Qty"] = pd.to_numeric(prod_df["Rejected_Qty"], errors='coerce')
    qual_df["Estimated_Loss_EGP"] = pd.to_numeric(qual_df["Estimated_Loss_EGP"], errors='coerce')
    
    avg_prod_efficiency = prod_df["Efficiency_Pct"].mean()
    total_rejected_qty = prod_df["Rejected_Qty"].sum()
    total_quality_loss = qual_df["Estimated_Loss_EGP"].sum()
    
    print(f"\n 3. Production and Quality:")
    print(f"  - Average Production Efficiency:             {avg_prod_efficiency:.2f}%")
    print(f"  - Total Rejected Quantity:    {total_rejected_qty:,.0f} pieces")
    print(f"  - Total Quality Loss (Estimated Loss): {total_quality_loss:,.2f} EGP")

# 4. تحليل التسويق والزيارات الإلكترونية (Marketing & Web Traffic)
if "Marketing" in cleaned_dfs and "Web_Traffic_Daily" in cleaned_dfs:
    mkt_df = cleaned_dfs["Marketing"].copy()
    web_df = cleaned_dfs["Web_Traffic_Daily"].copy()
    
    mkt_df["Spend_EGP"] = pd.to_numeric(mkt_df["Spend_EGP"], errors='coerce')
    mkt_df["Attributed_Revenue_EGP"] = pd.to_numeric(mkt_df["Attributed_Revenue_EGP"], errors='coerce')
    web_df["Conversion_Rate"] = pd.to_numeric(web_df["Conversion_Rate"], errors='coerce')
    
    total_mkt_spend = mkt_df["Spend_EGP"].sum()
    total_attr_rev = mkt_df["Attributed_Revenue_EGP"].sum()
    roas = total_attr_rev / total_mkt_spend if total_mkt_spend else 0
    avg_conversion = web_df["Conversion_Rate"].mean()
    
    print(f"\n 4. Marketing and Web Traffic:")
    print(f"  - Total Marketing Spend:             {total_mkt_spend:,.2f} EGP")
    print(f"  - Attributed Revenue:       {total_attr_rev:,.2f} EGP")
    print(f"  - Return on Ad Spend (ROAS):    {roas:.2f}x")
    print(f"  - Average Conversion Rate: {avg_conversion:.2f}%")

# 5. تحليل خدمة العملاء (Customer Service)
if "Customer_Service" in cleaned_dfs:
    cs_df = cleaned_dfs["Customer_Service"].copy()
    
    cs_df["CSAT_1_to_5"] = pd.to_numeric(cs_df["CSAT_1_to_5"], errors='coerce')
    cs_df["Resolution_Hours"] = pd.to_numeric(cs_df["Resolution_Hours"], errors='coerce')
    
    avg_csat = cs_df["CSAT_1_to_5"].mean()
    avg_res_hours = cs_df["Resolution_Hours"].mean()
    resolved_bool = cs_df["Resolved"].astype(str).str.upper().isin(["TRUE", "1", "YES"])
    resolved_pct = (resolved_bool.sum() / len(cs_df)) * 100
    
    print(f"\n 5. Customer Service:")
    print(f"  - Average Customer Satisfaction (CSAT):            {avg_csat:.2f} / 5.0")
    print(f"  - Average Resolution Time (Resolution Hours):   {avg_res_hours:.1f} hours")
    print(f"  - Resolution Rate (Resolved %):   {resolved_pct:.2f}%")

print("\n" + "=" * 60)

  (EDA Summary)



 1. Sales and financial performance:
  - Total Gross Sales: 600,092,061.00 EGP
  - Total Net Sales:   421,338,377.53 EGP
  - Total Gross Profit:   64,888,289.90 EGP
  - Average Gross Margin %:    15.40%
  - Return Rate %:        4.75%

 2. Inventory and supply chain:
  - Average Days of Cover: 335.3 days
  - Total Stockout Days: 1,197 days

 3. Production and Quality:
  - Average Production Efficiency:             82.21%
  - Total Rejected Quantity:    156,240 pieces
  - Total Quality Loss (Estimated Loss): 16,322,914.13 EGP

 4. Marketing and Web Traffic:
  - Total Marketing Spend:             14,583,976.00 EGP
  - Attributed Revenue:       179,741,957.27 EGP
  - Return on Ad Spend (ROAS):    12.32x
  - Average Conversion Rate: 0.03%

 5. Customer Service:
  - Average Customer Satisfaction (CSAT):            4.00 / 5.0
  - Average Resolution Time (Resolution Hours):   17.4 hours
  - Resolution Rate (Resolved %):   95.08%



In [12]:
print("=" * 65)
print(" (Deep-Dive Analysis)")
print("=" * 65)

# 1. تحليل أعلى الأسباب أداءً وخسارة في الجودة
if "Quality" in cleaned_dfs:
    qual_df = cleaned_dfs["Quality"].copy()
    qual_df["Estimated_Loss_EGP"] = pd.to_numeric(qual_df["Estimated_Loss_EGP"], errors='coerce')
    
    loss_by_root_cause = qual_df.groupby("Root_Cause")["Estimated_Loss_EGP"].sum().sort_values(ascending=False)
    print("\n Top reasons for material losses in quality (Root Cause Analysis):")
    print(loss_by_root_cause.apply(lambda x: f"{x:,.2f} EGP"))

# 2. تحليل المبيعات والربحية حسب المحافظات وقنوات البيع
if "Sales_Transactions" in cleaned_dfs:
    sales_df = cleaned_dfs["Sales_Transactions"].copy()
    sales_df["Net_Sales_EGP"] = pd.to_numeric(sales_df["Net_Sales_EGP"], errors='coerce')
    sales_df["Gross_Profit_EGP"] = pd.to_numeric(sales_df["Gross_Profit_EGP"], errors='coerce')
    
    sales_by_channel = sales_df.groupby("Channel")[["Net_Sales_EGP", "Gross_Profit_EGP"]].sum()
    sales_by_channel["Margin_%"] = (sales_by_channel["Gross_Profit_EGP"] / sales_by_channel["Net_Sales_EGP"]) * 100
    
    print("\n Financial Performance by Sales Channel (Channel Performance):")
    print(sales_by_channel.applymap(lambda x: f"{x:,.2f}" if isinstance(x, (int, float)) and x > 100 else f"{x:.2f}%"))

# 3. تحليل أعطال الماكينات وتأثيرها على الإنتاج
if "Machine_Downtime" in cleaned_dfs:
    dt_df = cleaned_dfs["Machine_Downtime"].copy()
    dt_df["Downtime_Hours"] = pd.to_numeric(dt_df["Downtime_Hours"], errors='coerce')
    
    dt_by_reason = dt_df.groupby("Reason")["Downtime_Hours"].sum().sort_values(ascending=False)
    print("\n Top Reasons for Machine Downtime (Hours):")
    print(dt_by_reason.apply(lambda x: f"{x:,.1f} hours"))

print("\n" + "=" * 65)

 (Deep-Dive Analysis)

 Top reasons for material losses in quality (Root Cause Analysis):
Root_Cause
Curing temperature       2,181,639.29 EGP
Process variation        2,145,308.03 EGP
Handling / packing       2,047,728.11 EGP
Operator method          2,040,608.52 EGP
Machine calibration      2,031,247.46 EGP
Pattern / measurement    2,018,582.40 EGP
No significant issue     1,994,872.65 EGP
Material variation       1,862,927.67 EGP
Name: Estimated_Loss_EGP, dtype: object

 Financial Performance by Sales Channel (Channel Performance):
            Net_Sales_EGP Gross_Profit_EGP Margin_%
Channel                                            
Online      32,629,574.45    11,384,070.74   34.89%
Retail      29,347,687.40    10,869,176.78   37.04%
Wholesale  359,361,115.68    42,635,042.38   11.86%

 Top Reasons for Machine Downtime (Hours):
Reason
Mechanical fault       1,530.9 hours
Setup / changeover     1,284.8 hours
Planned maintenance    1,224.5 hours
Material wait            846.0 hours


C:\Users\kamel\AppData\Local\Temp\ipykernel_20164\3375022262.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(sales_by_channel.applymap(lambda x: f"{x:,.2f}" if isinstance(x, (int, float)) and x > 100 else f"{x:.2f}%"))


In [13]:
!pip install streamlit plotly pandas


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\kamel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ------------